In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/digit-recognizer/sample_submission.csv
/kaggle/input/competitions/digit-recognizer/train.csv
/kaggle/input/competitions/digit-recognizer/test.csv


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
import os

#1-Load Data
train_path = "/kaggle/input/competitions/digit-recognizer/train.csv"
test_path = "/kaggle/input/competitions/digit-recognizer/test.csv"

# Check if files exist
print("Train file exists:", os.path.exists(train_path))
print("Test file exists:", os.path.exists(test_path))

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)

#2-Split features/label
X = train_df.drop("label",axis=1).values
y = train_df["label"].values
X_test = test_df.values

#3 Normalize and Reshape MNIST images are 28x28
X = X.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

X = X.reshape(-1,28,28,1)
X_test = X_test.reshape(-1,28,28,1)

#4-Train/Validation Split
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.1, 
                                                    random_state=42, stratify=y)
print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)

#5-Build CNN Model
model = keras.Sequential([
    layers.Input(shape=(28,28,1)),
    
    layers.Conv2D(32, kernel_size=(3,3), activation="relu"),
    layers.BatchNormalization(),
    layers.Conv2D(32, kernel_size=(3,3), activation="relu"),
    layers.MaxPooling2D(pool_size=(2,2)),
    layers.Dropout(0.25),

    layers.Conv2D(64, kernel_size=(3,3), activation="relu"),
    layers.BatchNormalization(),
    layers.Conv2D(64, kernel_size=(3,3), activation="relu"),
    layers.MaxPooling2D(pool_size=(2,2)),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(10, activation="softmax")    
])

model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=1e-3),
    loss = "sparse_categorical_crossentropy",
    metrics = ["accuracy"]
)

model.summary()

#6-Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6        
    )
]

#7-Train
history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=20,
    batch_size=128,
    callbacks=callbacks,
    verbose=1
)

#8-Validate
val_loss, val_acc = model.evaluate(X_valid, y_valid, verbose=0)
print("Validation accuracy:", val_acc)

#9-Predict test set
test_probs = model.predict(X_test, verbose=0)
test_pred = np.argmax(test_probs, axis=1)

#10 Submission File (with verification)
print("\n" + "="*50)
print("Creating submission file...")
print("="*50)

submission = pd.DataFrame({
    "ImageId": np.arange(1, len(test_pred) + 1),
    "Label": test_pred
})

# Save with explicit path
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)

# Verify file was created
if os.path.exists(submission_path):
    file_size = os.path.getsize(submission_path)
    print(f"✓ SUCCESS: submission.csv created!")
    print(f"  Location: {os.path.abspath(submission_path)}")
    print(f"  File size: {file_size} bytes")
    print(f"  Number of predictions: {len(test_pred)}")
    print("\nPreview of submission file:")
    print(submission.head(10))
    print(f"\nLast 5 predictions: {test_pred[-5:]}")
else:
    print(f"✗ ERROR: submission.csv was NOT created!")
    print(f"  Current directory: {os.getcwd()}")
    print(f"  Files in directory: {os.listdir()}")

print("="*50)

# For Kaggle - optionally display download link
try:
    from IPython.display import FileLink, display
    display(FileLink('submission.csv'))
    print("\nClick the link above to download the submission file")
except:
    pass

2026-03-24 15:17:06.281127: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774365426.458010      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774365426.512666      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774365426.936358      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774365426.936404      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774365426.936408      24 computation_placer.cc:177] computation placer alr

Train file exists: True
Test file exists: True
Train Shape: (42000, 785)
Test Shape: (28000, 784)
X_train: (37800, 28, 28, 1)
X_valid: (4200, 28, 28, 1)


I0000 00:00:1774365455.665157      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1774365455.671183      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 26, 26, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 24, 24, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 12, 12, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 12, 12, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 10, 10, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 10, 10, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 8, 8, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 4, 4, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 4, 4, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

 16/296 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.2949 - loss: 2.4940

I0000 00:00:1774365465.820755      70 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


296/296 ━━━━━━━━━━━━━━━━━━━━ 16s 27ms/step - accuracy: 0.7766 - loss: 0.7409 - val_accuracy: 0.2545 - val_loss: 2.2007 - learning_rate: 0.0010
Epoch 2/20
296/296 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9664 - loss: 0.1089 - val_accuracy: 0.9802 - val_loss: 0.0667 - learning_rate: 0.0010
Epoch 3/20
296/296 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9766 - loss: 0.0742 - val_accuracy: 0.9871 - val_loss: 0.0400 - learning_rate: 0.0010
Epoch 4/20
296/296 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9821 - loss: 0.0582 - val_accuracy: 0.9879 - val_loss: 0.0373 - learning_rate: 0.0010
Epoch 5/20
296/296 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9835 - loss: 0.0504 - val_accuracy: 0.9893 - val_loss: 0.0357 - learning_rate: 0.0010
Epoch 6/20
296/296 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9856 - loss: 0.0456 - val_accuracy: 0.9907 - val_loss: 0.0319 - learning_rate: 0.0010
Epoch 7/20
296/296 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9875 - loss: 0.0408 - val_accur

/kaggle/working/submission.csv


Click the link above to download the submission file
